# SPAD photon-stream search on T4 x2: batch 3 (fused step, larger search)

Attach the private `spad-stream-design` dataset, choose **GPU T4 x2**, then use **Save Version > Save & Run All**.
A saved version keeps running after the browser closes. Download `spad_runs_batch3.zip` from the output when it finishes.

To continue an unfinished sweep, attach the previous output as an input and set `PREVIOUS` below.

Batch 3: fused gradient step, compiled bit-packed search, population 512, 1500 generations. Upload `spad-stream-design.zip` from this folder as a **new version** of the dataset first; older bundles lack these changes.

In [ ]:
import glob, os, shutil, subprocess, sys

matches = glob.glob('/kaggle/input/**/code/spad/training/train_stream.py', recursive=True)
assert matches, 'Attach the spad-stream-design dataset'
BUNDLE = matches[0].rsplit('/code/', 1)[0]
WORK = '/kaggle/working'
CODE, DATA, RUNS = f'{WORK}/code', f'{BUNDLE}/data', f'{WORK}/runs'
SESSION_SECONDS = 8 * 3600   # stop safely below Kaggle's session limit
PREVIOUS = None              # for example '/kaggle/input/previous-output/runs'

shutil.copytree(f'{BUNDLE}/code', CODE, dirs_exist_ok=True)
if PREVIOUS:
    shutil.copytree(PREVIOUS, RUNS, dirs_exist_ok=True)
import json
splits = {k: v['split'] for k, v in json.load(open(f'{DATA}/processed/metadata.json'))['captures'].items()}
held_out = [k for k, v in splits.items() if v not in ('train', 'architecture', 'validation')]
held_out += [p for p in os.listdir(f'{DATA}/processed') if p.split('.')[0].removeprefix('labels_') in {'7', '8', '9', '10'} or p.startswith(('parking', 'labels_parking'))]
assert not held_out, f'Held-out captures must not be here: {held_out}'
print(BUNDLE)

In [ ]:
import torch
assert torch.cuda.is_available(), 'Choose GPU T4 x2 under Accelerator'
GPUS = torch.cuda.device_count()
print(torch.__version__, GPUS, [torch.cuda.get_device_name(i) for i in range(GPUS)])

CPU and CUDA must agree exactly. Stop here if this fails.

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'tests.test_device', '-v'], cwd=CODE, check=True)

Pick the fastest settings on this GPU before spending quota. Each `torch.compile` mode is timed against plain PyTorch for the gradient step and for the exact search; a mode that fails is skipped. The fastest of each is used for training.

In [ ]:
import json
os.makedirs(RUNS, exist_ok=True)

def bench(*flags):
    done = subprocess.run([sys.executable, '-m', 'spad.evaluation.bench_stream', '--device', 'cuda', *flags],
                          cwd=CODE, capture_output=True, text=True)
    if done.returncode or 'torch.compile' in done.stdout.split('{')[0]:
        print('  skipped', flags, done.stderr[-300:])
        return None
    return json.loads(done.stdout[done.stdout.index('{'):])

report = {'gradient': {}, 'search': {}}
for mode in ('none', 'default', 'reduce-overhead'):
    result = bench('--compile', mode, '--batches', '1024', '--skip-search', '--repeats', '2')
    if result:
        report['gradient'][mode] = result['gradient']['1024']['seconds_per_step']
        print(f"gradient  compile={mode:16s} {report['gradient'][mode]:.3f} s per pass (batch 1024)")
for mode in ('none', 'default', 'reduce-overhead'):
    result = bench('--compile-search', mode, '--populations', '256', '--search-rays', '2048', '--skip-gradient', '--repeats', '2')
    if result:
        report['search'][mode] = result['hard_search']['256']['circuits_per_second']
        print(f"search    compile={mode:16s} {report['search'][mode]:.0f} circuits per second (2048 rays)")
GRAD_COMPILE = min(report['gradient'], key=report['gradient'].get)
SEARCH_COMPILE = max(report['search'], key=report['search'].get)
report['chosen'] = {'gradient': GRAD_COMPILE, 'search': SEARCH_COMPILE}
print('using', report['chosen'])
open(f'{RUNS}/benchmark_t4.json', 'w').write(json.dumps(report, indent=2))

In [ ]:
# Larger search than the 2026-11-08 batch (population 128, 600 generations), made affordable
# by the fused gradient step and the compiled, bit-packed exact search.
BATCH, POPULATION, SEARCH_RAYS, GENERATIONS = 1024, 512, 2048, 1500
subprocess.run([sys.executable, 'kaggle/launch.py', '--jobs', 'kaggle/jobs.txt', '--out', RUNS, '--data', DATA,
                '--gpus', str(GPUS), '--max-seconds', str(SESSION_SECONDS),
                '--batch-size', str(BATCH), '--population', str(POPULATION),
                '--search-rays', str(SEARCH_RAYS), '--generations', str(GENERATIONS),
                '--compile', GRAD_COMPILE, '--compile-search', SEARCH_COMPILE],
               cwd=CODE, check=True)

In [ ]:
import json
rows = []
for path in glob.glob(f'{RUNS}/*/summary.json'):
    s = json.load(open(path))
    rows.append((s['validation_fresh_photons']['mae_m'], os.path.basename(os.path.dirname(path)), s['seconds']))
for mae, name, seconds in sorted(rows):
    print(f'{name:22s} MAE {mae:.3f} m  {seconds/60:.1f} min')
shutil.make_archive(f'{WORK}/spad_runs_batch3', 'zip', RUNS)